In [0]:
SELECT COUNT(DISTINCT canonicalCustomerName) as total_customers 
FROM main.field_labs.lakebridge_logs
WHERE canonicalCustomerName IS NOT NULL 
  AND canonicalCustomerName <> 'Databricks' 
  AND canonicalCustomerName <> ''

## Workspaces per cloud


In [0]:
SELECT
  salesforce_account_id,
  COALESCE(`AWS`, 0) AS AWS,
  COALESCE(`Azure`, 0) AS Azure,
  COALESCE(`GCP`, 0) AS GCP
FROM (
  SELECT
    salesforce_account_id,
    cloud_type,
    COUNT(DISTINCT workspace_id) AS workspace_count
  FROM main.certified.workspaces_latest
  WHERE salesforce_account_id IS NOT NULL
  GROUP BY salesforce_account_id, cloud_type
) src
PIVOT (
  SUM(workspace_count) FOR cloud_type IN ('aws' AS AWS, 'azure' AS Azure, 'gcp' AS GCP)
);

In [0]:
  SELECT
    salesforce_account_id,
    cloud_type,
    COUNT(DISTINCT workspace_id) AS workspace_count
  FROM main.certified.workspaces_latest
  WHERE salesforce_account_id IS NOT NULL
  GROUP BY salesforce_account_id, cloud_type

In [0]:
SELECT *
FROM (
  SELECT
    salesforce_account_id,
    cloud_type,
    COUNT(DISTINCT workspace_id) AS workspace_count
  FROM main.certified.workspaces_latest
  WHERE salesforce_account_id IS NOT NULL
  GROUP BY salesforce_account_id, cloud_type
) src
PIVOT (
  SUM(workspace_count) FOR cloud_type IN ('AWS' AS AWS, 'Azure' AS Azure, 'GCP' AS GCP)
);


## Users using Serverless

In [0]:
-- Parameters
-- Adjust the lookback_days as needed (e.g., 30, 90)
WITH params AS (
  SELECT 30 AS lookback_days
),
base AS (
  SELECT
    account_id,
    workspace_id,
    CAST(usage_date AS DATE) AS usage_dt,
    -- Heuristic extraction of the user identity (adjust to your schema)
    COALESCE(
      usage_metadata.user_email,
      usage_metadata.run_as_user_email,
      usage_metadata.owner_user_email,
      usage_metadata.principal_email,
      CAST(usage_metadata.user_id AS STRING),
      CAST(usage_metadata.principal_id AS STRING)
    ) AS user_identity
  FROM main.centralized_system_tables.billing_usage
  WHERE product_features.is_serverless = TRUE
    AND usage_date >= DATE_SUB(CURRENT_DATE(), (SELECT lookback_days FROM params))
)
SELECT
  account_id,
  COUNT(DISTINCT user_identity) AS serverless_users
FROM base
WHERE user_identity IS NOT NULL AND user_identity <> ''
GROUP BY account_id
ORDER BY serverless_users DESC;

## System Tables queries

In [0]:
-- Count System Tables queries per Salesforce account, last 30 days
WITH workspace_account AS (
  SELECT workspace_id, salesforce_account_id
  FROM main.certified.workspaces_latest
),
qh AS (
  SELECT
    workspace_id,
    /* Use the correct time column in your environment: query_start_time or start_time */
    query_start_time,
    /* Prefer relation/schema metadata fields if present (e.g., referenced_schema, referenced_relation) */
    query_text
  FROM main.centralized_system_tables.query_history
)
SELECT
  ws.salesforce_account_id,
  COUNT(*) AS num_system_table_queries
FROM qh
JOIN ws USING (workspace_id)
WHERE query_start_time >= current_date - INTERVAL 30 DAYS
  /* Prefer schema-level metadata if available:
     AND referenced_schema = 'system'
     OR referenced_relation LIKE 'system.%'
     Fallback (text heuristic) if relation metadata isn’t present: */
  AND LOWER(query_text) LIKE '% system.%'
GROUP BY ws.salesforce_account_id
ORDER BY num_system_table_queries DESC;

In [0]:
-- Count System Tables queries per Salesforce account in the last 30 days

-- Workspace → Salesforce account mapping
WITH ws AS (
  SELECT workspace_id, salesforce_account_id
  FROM main.certified.workspaces_latest
  WHERE inferred_workspace_type IN ('External','MicrosoftPaid')
),

-- Query history (pick one source depending on environment):
-- 1) Customer account: system.query.history
-- 2) Central mirror:   main.centralized_system_tables.query_history
qh AS (
  SELECT
    workspace_id,
    statement_id,
    start_time,
    statement_text
  FROM system.query.history  -- or main.centralized_system_tables.query_history
  WHERE start_time >= current_date - INTERVAL 30 DAYS
),

-- Lineage-backed detection of system.* references
sys_lineage AS (
  SELECT
    l.workspace_id,
    /* Use cache_origin_statement_id if present for cached queries; else statement_id */
    COALESCE(qh.statement_id /* replace with qh.cache_origin_statement_id when available */, qh.statement_id) AS sid
  FROM qh
  JOIN system.access.table_lineage l
    ON l.statement_id = qh.statement_id
  WHERE LOWER(l.source_table_full_name) LIKE 'system.%'
),

-- Text-backed fallback when lineage isn’t populated
sys_text AS (
  SELECT workspace_id, statement_id AS sid
  FROM qh
  WHERE statement_text IS NOT NULL
    AND LOWER(statement_text) LIKE '% system.%'
),

-- Union and roll up by SFDC account
sys_queries AS (
  SELECT * FROM sys_lineage
  UNION ALL
  SELECT * FROM sys_text
)
SELECT
  ws.salesforce_account_id,
  COUNT(DISTINCT sys_queries.sid) AS num_system_table_queries
FROM sys_queries
JOIN ws USING (workspace_id)
GROUP BY ws.salesforce_account_id
ORDER BY num_system_table_queries DESC;

## Jobs using serverless

In [0]:
WITH workspace_account AS (
  SELECT workspace_id, salesforce_account_id
  FROM main.certified.workspaces_latest
),

job_data as (
  SELECT workspace_id, usage_metadata.job_id 
  FROM main.centralized_system_tables.billing_usage
  WHERE product_features.is_serverless = TRUE
      AND billing_origin_product = 'JOBS'                  -- exclude non-Job features billed under serverless SKU
      AND usage_metadata.job_id IS NOT NULL

)

SELECT wa.salesforce_account_id,
       COUNT(DISTINCT j.job_id) AS num_serverless_jobs
FROM workspace_account wa
LEFT JOIN job_data j on wa.workspace_id = j.workspace_id
GROUP BY wa.salesforce_account_id
ORDER BY num_serverless_jobs DESC;



## Genie

In [0]:
SHOW TABLE EXTENDED LIKE 'main.eng_datarooms.genie_logs_metric_view_2'

In [0]:
SELECT * FROM main.eng_datarooms.genie_logs_metric_view_2;


In [0]:
genie_logs_metric_view_2

In [0]:
SELECT DISTINCT unified_user_agent.product
FROM main.eng_deco_usage.deco_usage_logs_normalized

## For apps

In [0]:
WITH app_data as(
  SELECT salesforce_account_name, COUNT(DISTINCT(app_id)) as total_apps
  FROM users.jerry_liang.databricks_apps_daily_counts
  WHERE partition_date = (SELECT MAX(partition_date) FROM users.jerry_liang.databricks_apps_daily_counts)
  --AND state = 'active'
  GROUP BY salesforce_account_name
)

SELECT * FROM app_data ORDER BY salesforce_account_name;



## For connections, external locations, managed tables, managed volumes and storage credentials

In [0]:
SELECT * FROM main.eng_data_gov_obs.mirror_table_mc_volumes

In [0]:
WITH account_data as (
  SELECT account_id, account_name
  FROM main.gtm_silver.account_dim
 WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM main.gtm_silver.account_dim)
),

workspaces_cloud as (
  SELECT
    salesforce_account_id,
    COALESCE(`AWS`, 0) AS AWS_workspaces,
    COALESCE(`Azure`, 0) AS Azure_workspaces,
    COALESCE(`GCP`, 0) AS GCP_workspaces
  FROM (
    SELECT
      salesforce_account_id,
      cloud_type,
      COUNT(DISTINCT workspace_id) AS workspace_count
    FROM main.certified.workspaces_latest
    WHERE salesforce_account_id IS NOT NULL
    GROUP BY salesforce_account_id, cloud_type
  ) src
  PIVOT (
    SUM(workspace_count) FOR cloud_type IN ('aws' AS AWS, 'azure' AS Azure, 'gcp' AS GCP)
  )
  
),

app_data as(
  SELECT salesforce_account_name, COUNT(DISTINCT(app_id)) as total_apps
  FROM users.jerry_liang.databricks_apps_daily_counts
  WHERE partition_date = (SELECT MAX(partition_date) FROM users.jerry_liang.databricks_apps_daily_counts)
  --AND state = 'active'
  GROUP BY salesforce_account_name
),

account_metastore AS (
  SELECT salesforce_account_id, UNHEX(metastore_id) as metastore_id
  FROM main.eng_data_gov_obs.metastores
),

storage_credentials_data as (
  SELECT metastore_id, COUNT(DISTINCT(id)) as total_storage_credentials
  FROM main.eng_data_gov_obs.mirror_table_mc_storage_credentials
  GROUP BY metastore_id
  --WHERE is_deleted = false
),

external_location_data as (
  SELECT metastore_id, COUNT(DISTINCT(id)) as total_external_locations
  FROM main.eng_data_gov_obs.mirror_table_mc_external_locations
  GROUP BY metastore_id
  --WHERE is_deleted = false
),

managed_tables_data as (
  SELECT metastore_id, COUNT(DISTINCT(id)) as total_managed_tables
  FROM main.eng_data_gov_obs.mirror_table_mc_tables
  WHERE type = 'MANAGED'
  GROUP BY metastore_id
  --WHERE is_deleted = false
),

managed_volumes_data as (
  SELECT metastore_id, COUNT(DISTINCT(id)) as total_managed_volumes
  FROM main.eng_data_gov_obs.mirror_table_mc_volumes
  WHERE type = 'MANAGED'
  GROUP BY metastore_id
  --WHERE is_deleted = false
),

connection_data as (
  SELECT metastore_id, COUNT(DISTINCT(id)) as total_connections_created
  FROM main.eng_data_gov_obs.mirror_table_mc_connections
  GROUP BY metastore_id
  --WHERE is_deleted = false
)

SELECT ac.account_id, ac.account_name, COALESCE(total_apps,0) as total_apps, COALESCE(SUM(total_connections_created),0) as total_connections, COALESCE(SUM(total_external_locations),0) as total_external_locations, COALESCE(SUM(total_managed_tables),0) as total_managed_tables, COALESCE(SUM(total_managed_volumes),0) as total_managed_volumes, COALESCE(SUM(total_storage_credentials),0) as total_storage_credentials, aws_workspaces, azure_workspaces, gcp_workspaces
FROM account_data ac
LEFT JOIN account_metastore m
ON ac.account_id = m.salesforce_account_id
LEFT JOIN connection_data c
ON m.metastore_id = c.metastore_id
LEFT JOIN external_location_data el
ON m.metastore_id = el.metastore_id
LEFT JOIN managed_tables_data mt
ON m.metastore_id = mt.metastore_id
LEFT JOIN managed_volumes_data mv
ON m.metastore_id = mv.metastore_id
LEFT JOIN storage_credentials_data sc
ON m.metastore_id = sc.metastore_id
LEFT JOIN app_data a
ON ac.account_name = a.salesforce_account_name
LEFT JOIN workspaces_cloud w
ON ac.account_id = w.salesforce_account_id
GROUP BY ac.account_id, ac.account_name, total_apps, aws_workspaces, azure_workspaces, gcp_workspaces;


In [0]:
WITH account_data as (
  SELECT account_id, account_name
  FROM main.gtm_silver.account_dim
 WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM main.gtm_silver.account_dim)
),

app_data as(
  SELECT salesforce_account_name, COUNT(DISTINCT(app_id)) as total_apps
  FROM users.jerry_liang.databricks_apps_daily_counts
  WHERE partition_date = (SELECT MAX(partition_date) FROM users.jerry_liang.databricks_apps_daily_counts)
  --AND state = 'active'
  GROUP BY salesforce_account_name
),

account_metastore AS (
  SELECT salesforce_account_id, UNHEX(metastore_id) as metastore_id
  FROM main.eng_data_gov_obs.metastores
),

storage_credentials_data as (
  SELECT metastore_id, COUNT(DISTINCT(id)) as total_storage_credentials
  FROM main.eng_data_gov_obs.mirror_table_mc_storage_credentials
  GROUP BY metastore_id
  --WHERE is_deleted = false
),

external_location_data as (
  SELECT metastore_id, COUNT(DISTINCT(id)) as total_external_locations
  FROM main.eng_data_gov_obs.mirror_table_mc_external_locations
  GROUP BY metastore_id
  --WHERE is_deleted = false
),

managed_tables_data as (
  SELECT metastore_id, COUNT(DISTINCT(id)) as total_managed_tables
  FROM main.eng_data_gov_obs.mirror_table_mc_tables
  WHERE type = 'MANAGED'
  GROUP BY metastore_id
  --WHERE is_deleted = false
),

managed_volumes_data as (
  SELECT metastore_id, COUNT(DISTINCT(id)) as total_managed_volumes
  FROM main.eng_data_gov_obs.mirror_table_mc_volumes
  WHERE type = 'MANAGED'
  GROUP BY metastore_id
  --WHERE is_deleted = false
),

connection_data as (
  SELECT metastore_id, COUNT(DISTINCT(id)) as total_connections_created
  FROM main.eng_data_gov_obs.mirror_table_mc_connections
  GROUP BY metastore_id
  --WHERE is_deleted = false
)

SELECT ac.account_id, ac.account_name, COALESCE(SUM(total_connections_created),0) as total_connections, COALESCE(SUM(total_external_locations),0) as total_external_locations, COALESCE(SUM(total_managed_tables),0) as total_managed_tables, COALESCE(SUM(total_managed_volumes),0) as total_managed_volumes, COALESCE(SUM(total_storage_credentials),0) as total_storage_credentials, total_apps
FROM account_data ac
LEFT JOIN account_metastore m
ON ac.account_id = m.salesforce_account_id
LEFT JOIN connection_data c
ON m.metastore_id = c.metastore_id
LEFT JOIN external_location_data el
ON m.metastore_id = el.metastore_id
LEFT JOIN managed_tables_data mt
ON m.metastore_id = mt.metastore_id
LEFT JOIN managed_volumes_data mv
ON m.metastore_id = mv.metastore_id
LEFT JOIN storage_credentials_data sc
ON m.metastore_id = sc.metastore_id
LEFT JOIN app_data a
ON ac.account_name = a.salesforce_account_name
GROUP BY ac.account_id, ac.account_name, total_apps;

In [0]:
SELECT * FROM main.eng_aibi.lakeview_usage_customer_rollup LIMIT 10;



SELECT * FROM main.eng_aibi.data_usage_log;


## For Unity Catalog

In [0]:
WITH combined_catalogs AS (
  SELECT 
    metastore_id,
    type
  FROM main.data_centralized_db_snapshot.managedcatalog__mysql__mc_catalogs_latest_snapshot
  
  UNION ALL
  
  SELECT 
    metastore_id,
    type
  FROM main.data_centralized_db_snapshot.managedcatalog__tidb__mc_catalogs_latest_snapshot
)

SELECT * FROM combined_catalogs

In [0]:
WITH combined_catalogs AS (
  SELECT 
    metastore_id,
    type
  FROM main.data_centralized_db_snapshot.managedcatalog__mysql__mc_catalogs_latest_snapshot
  
  UNION ALL
  
  SELECT 
    metastore_id,
    type
  FROM main.data_centralized_db_snapshot.managedcatalog__tidb__mc_catalogs_latest_snapshot
)
SELECT 
  salesforce_account_id, 
  COUNT(CASE WHEN catalogs.type = 'MANAGED_CATALOG' THEN 1 END) as number_of_managed_catalogs,
  COUNT(CASE WHEN catalogs.type = 'MANAGED_ONLINE_CATALOG' THEN 1 END) as number_of_managed_online_catalogs,
  COUNT(CASE WHEN catalogs.type = 'FOREIGN_CATALOG' THEN 1 END) as number_of_foreign_catalogs,
  COUNT(CASE WHEN catalogs.type = 'SYSTEM_CATALOG' THEN 1 END) as number_of_system_catalogs,
  COUNT(CASE WHEN catalogs.type = 'DELTASHARING_CATALOG' THEN 1 END) as number_of_deltasharing_catalogs,
  COUNT(CASE WHEN catalogs.type = 'INTERNAL_CATALOG' THEN 1 END) as number_of_internal_catalogs
FROM combined_catalogs catalogs
LEFT JOIN main.eng_data_gov_obs.metastores metastores
ON catalogs.metastore_id = UNHEX(metastores.metastore_id)
GROUP BY salesforce_account_id

In [0]:
SELECT * 
FROM main.eng_data_gov_obs.mirror_table_mc_external_locations ;

In [0]:
SELECT * FROM main.eng_data_gov_obs.metastores metastores LIMIT 10;